# Notebook 1 — Explore the Berka bank database

| What | Where |
|---|---|
| Database | `data/financial.sqlite`, with 8 tables and about 1 million transactions |
| Source | [domyn/FINCH](https://huggingface.co/datasets/domyn/FINCH), the BIRD database named `financial` |
| Origin | Petr Berka's Czech bank data from PKDD 1999 |
| Licence | CC BY-NC 4.0 |
| Column descriptions | `semantic_layer.json` |

Try to answer “How many accounts are in Pisek?” First find what `A2` means, then check how district names are spelled in the database. Use the same approach for statement frequency, which is stored as a Czech code such as `POPLATEK PO OBRATU`.


## Step 0 — Setup

Ollama is not required in this notebook. The SQL runs locally.

If `data/financial.sqlite` is missing, run `python download_corpus.py` in this folder. That copies one 71 MB file out of FINCH. The other databases in that dataset stay in the Hugging Face cache.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "text2sql.py").exists():
    raise SystemExit("Set the notebook's working directory to 17_agentic_text2sql.")
sys.path.insert(0, str(HERE))

import text2sql as desk

connection = desk.connect()
tables = [row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1")]
print("tables:", ", ".join(tables))
for name in tables:
    count = connection.execute(f"SELECT COUNT(*) FROM {desk.qtable(name)}").fetchone()[0]
    print(f"  {name:10} {count:,} rows")
connection.close()


tables: account, card, client, disp, district, loan, order, trans
  account    4,500 rows
  card       892 rows
  client     5,369 rows
  disp       5,369 rows
  district   77 rows
  loan       682 rows
  order      6,471 rows
  trans      1,056,320 rows


## 1. Look up the column names

Open `semantic_layer.json` and find `district.A11`. The column description says it holds the average salary. A model needs that description to choose the right column.

A district lookup uses the district table. The transaction table has 1,056,320 rows, and this question can be answered without opening it.


In [2]:
layer = desk.load_layer()
for column, meaning in layer["tables"]["district"]["columns"].items():
    print(f"{column:12} {meaning}")

print()
catalog = desk.value_catalog()
print("account.frequency")
for line in catalog["account.frequency"]:
    print(" ", line)
print("district.A3")
for line in catalog["district.A3"]:
    print(" ", line)


district_id  Primary key. Branch and client tables point here.
A2           District name, for example Pisek or Hl.m. Praha.
A3           Region name, for example east Bohemia or Prague.
A4           Number of inhabitants. Stored as text in this file.
A5           Municipalities with fewer than 499 inhabitants.
A6           Municipalities with 500 to 1999 inhabitants.
A7           Municipalities with 2000 to 9999 inhabitants.
A8           Municipalities with 10000 or more inhabitants.
A9           Number of cities.
A10          Ratio of urban inhabitants.
A11          Average salary.
A12          Unemployment rate in 1995.
A13          Unemployment rate in 1996.
A14          Entrepreneurs per 1000 inhabitants.
A15          Number of crimes committed in 1995.
A16          Number of crimes committed in 1996.

account.frequency
  POPLATEK MESICNE = monthly statement
  POPLATEK TYDNE = weekly statement
  POPLATEK PO OBRATU = statement issued after each transaction
district.A3
  south Morav

## 2. Check the spelling in the database

Run both filters and compare their counts. FINCH uses `east bohemia` and `poplatek po obratu`. This SQLite file stores `east Bohemia` and `POPLATEK PO OBRATU`. The comparison used here is case-sensitive.

If a count comes back as zero, look up the stored values and rerun the filter with the exact spelling.


In [3]:
published = """
SELECT COUNT(*) AS accounts
FROM account AS a
INNER JOIN district AS d ON a.district_id = d.district_id
WHERE d.A3 = 'east bohemia' AND a.frequency = 'poplatek po obratu'
"""
stored = """
SELECT COUNT(*) AS accounts
FROM account AS a
INNER JOIN district AS d ON a.district_id = d.district_id
WHERE d.A3 = 'east Bohemia' AND a.frequency = 'POPLATEK PO OBRATU'
"""
print("published spelling:", desk.execute_sql(published)["rows"])
print("stored spelling:   ", desk.execute_sql(stored)["rows"])


published spelling: [[0]]
stored spelling:    [[13]]


## 3. Filter by year

Dates in this file look like `1993-01-01`. In SQLite, `strftime('%Y', date)` returns the four-digit year as text. `%y` is unsupported and returns `NULL`, so comparing that result with `'1993'` matches zero rows. Run both queries and compare the counts. [SQLite date functions](https://www.sqlite.org/lang_datefunc.html)


In [4]:
print("opened in 1993 with %Y:", desk.execute_sql("SELECT COUNT(*) FROM account WHERE strftime('%Y', date) = '1993'")["rows"])
print("same filter with %y:   ", desk.execute_sql("SELECT COUNT(*) FROM account WHERE strftime('%y', date) = '1993'")["rows"])


opened in 1993 with %Y: [[1139]]
same filter with %y:    [[0]]
